# 🫁 Pneumonia Detection from Chest X-RaysBinary classification of chest X-ray images (NORMAL vs PNEUMONIA)  using a Custom CNN baseline and DenseNet121 transfer learning.---## 📋 Project Outline| Phase | Description ||-------|-------------|| **Phase 1** | Environment Setup & Imports || **Phase 2** | Data Loading & Exploration || **Phase 3** | Preprocessing & Data Splits || **Phase 4** | Data Generators & Augmentation || **Phase 5** | Custom CNN — Build, Train & Evaluate || **Phase 6** | DenseNet121 — Transfer Learning (Frozen Base) || **Phase 7** | DenseNet121 — Fine-Tuning (Unfreeze Top Layers) || **Phase 8** | Final Evaluation & Model Export |

## Phase 1 — Environment Setup & Imports

In [ ]:
import osimport warningsimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport tensorflow as tffrom sklearn.model_selection import train_test_splitfrom sklearn.utils.class_weight import compute_class_weightfrom sklearn.metrics import (    confusion_matrix, accuracy_score,    f1_score, precision_score, recall_score)from tensorflow.keras import layersfrom tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpointfrom tensorflow.keras.applications.densenet import preprocess_inputwarnings.filterwarnings('ignore')# Confirm GPU availabilityprint("TF version :", tf.__version__)print("GPUs found  :", tf.config.list_physical_devices('GPU'))

In [ ]:
# Verify Kaggle input directory contentsprint(os.listdir('/kaggle/input'))

## Phase 2 — Data Loading & Exploration

In [ ]:
BASE_PATH = '/kaggle/input/datasets/tolgadincer/labeled-chest-xray-images/chest_xray'# Confirm expected split folders existprint(os.listdir(BASE_PATH))

In [ ]:
def extract_df(path_):    """Walk a NORMAL/PNEUMONIA folder structure and return a DataFrame    with columns [img_paths, label]."""    img_paths, labels = [], []    for label in os.listdir(path_):        for fname in os.listdir(os.path.join(path_, label)):            img_paths.append(os.path.join(path_, label, fname))            labels.append(label)    return pd.DataFrame({"img_paths": img_paths, "label": labels})

In [ ]:
TRAIN_PATH = os.path.join(BASE_PATH, 'train')TEST_PATH  = os.path.join(BASE_PATH, 'test')train_df = extract_df(TRAIN_PATH)test_df  = extract_df(TEST_PATH)print("Train shape :", train_df.shape)print("Test shape  :", test_df.shape)train_df.head()

### 2.1 Sample Images

In [ ]:
def show_samples(df, num_images=10, title="Sample X-Ray Images"):    """Display a random grid of X-ray images with their class labels."""    plt.figure(figsize=(15, 6))    for i in range(num_images):        idx = np.random.randint(0, len(df))        img = plt.imread(df.iloc[idx, 0])        plt.subplot(2, 5, i + 1)        plt.imshow(img, cmap="gray")        plt.title(df.iloc[idx, 1], fontsize=9)        plt.axis("off")    plt.suptitle(title, fontsize=13, fontweight="bold")    plt.tight_layout()    plt.show()show_samples(train_df, title="Train — Sample X-Ray Images")show_samples(test_df,  title="Test  — Sample X-Ray Images")

### 2.2 Class Distribution

In [ ]:
def plot_distribution(train_df, test_df):    """Bar charts showing class balance for train and test sets."""    fig, axes = plt.subplots(1, 2, figsize=(14, 4))    for ax, df, title in zip(axes,                              [train_df, test_df],                              ["Train Set", "Test Set"]):        counts = df["label"].value_counts()        bars = ax.bar(counts.index, counts.values, color=["steelblue", "tomato"])        ax.set_title(title)        ax.set_xlabel("Class")        ax.set_ylabel("Count")        # Annotate bar heights        for bar, v in zip(bars, counts.values):            ax.text(bar.get_x() + bar.get_width() / 2, v + 20,                    str(v), ha="center", fontweight="bold")    plt.suptitle("Class Distribution — Pneumonia Dataset", fontsize=13, fontweight="bold")    plt.tight_layout()    plt.show()plot_distribution(train_df, test_df)

### 2.3 Image Properties

In [ ]:
# Sample 200 images to check resolution variabilitysample_df = train_df.sample(200, random_state=42)sizes = [plt.imread(p).shape[:2] for p in sample_df["img_paths"]]sizes_df = pd.DataFrame(sizes, columns=["height", "width"])print("Resolution statistics (200-image sample):")print(sizes_df.describe())print(f"\nUnique resolutions: {sizes_df.drop_duplicates().shape[0]} / {len(sizes_df)}")

In [ ]:
# Inspect a single image for dtype and pixel rangesample_img = plt.imread(train_df.iloc[0, 0])print("Shape :", sample_img.shape)print("Dtype :", sample_img.dtype)print("Min   :", sample_img.min(), "| Max:", sample_img.max())

In [ ]:
# Count how many images are truly grayscale vs. RGB across 200 samplesmode_counts = {"grayscale": 0, "rgb": 0, "other": 0}for path in sample_df["img_paths"]:    img = plt.imread(path)    if img.ndim == 2:        mode_counts["grayscale"] += 1    elif img.ndim == 3 and img.shape[2] == 3:        mode_counts["rgb"] += 1    else:        mode_counts["other"] += 1print("Channel mode counts:", mode_counts)

## Phase 3 — Preprocessing & Data Splits

In [ ]:
# Carve out a stratified 20% validation set from the training datatrain_df, val_df = train_test_split(    train_df,    test_size=0.20,    random_state=42,    stratify=train_df['label']   # preserve class ratio)print("Split sizes:")print(f"  Train : {train_df.shape[0]:>5}")print(f"  Val   : {val_df.shape[0]:>5}")print(f"  Test  : {test_df.shape[0]:>5}")print("\nTrain class balance:")print(train_df["label"].value_counts())print("\nVal class balance:")print(val_df["label"].value_counts())

## Phase 4 — Data Generators & Augmentation> **Note:** Augmentation is applied **only** to the training generator.  > Validation and test sets use raw (rescaled) images so they reflect real-world conditions.

In [ ]:
# ── Custom-CNN generators (simple rescale) ───────────────────────────────────train_gen = tf.keras.preprocessing.image.ImageDataGenerator(    rescale=1/255,    rotation_range=10,    zoom_range=0.1,    width_shift_range=0.1,    height_shift_range=0.1,    horizontal_flip=True       # horizontal flip is safe for X-rays)val_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1/255)# Flow from DataFrames — color_mode='rgb' ensures 3-channel input for the CNNBATCH = 32IMG_SIZE = (224, 224)train_data = train_gen.flow_from_dataframe(    train_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=BATCH, color_mode='rgb', seed=42)val_data = val_gen.flow_from_dataframe(    val_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=BATCH, color_mode='rgb', seed=42)test_data = val_gen.flow_from_dataframe(    test_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=1, color_mode='rgb', shuffle=False   # no shuffle for evaluation)labels_list = list(train_data.class_indices.keys())print("Class indices:", train_data.class_indices)

In [ ]:
# Visualise one augmented training batchimages, labels = next(train_data)plt.figure(figsize=(15, 6))for i in range(10):    plt.subplot(2, 5, i + 1)    plt.imshow(images[i])    plt.title(labels_list[int(labels[i])], fontsize=10)    plt.axis("off")plt.suptitle("Augmented Training Batch", fontsize=13, fontweight="bold")plt.tight_layout()plt.show()

## Phase 5 — Custom CNN — Build, Train & EvaluateA lightweight 3-block convolutional network used as a **baseline** before moving to transfer learning.

In [ ]:
# ── Architecture ─────────────────────────────────────────────────────────────inputs = layers.Input(shape=(224, 224, 3))x = layers.Conv2D(32,  (3, 3), activation="relu")(inputs)x = layers.MaxPooling2D()(x)x = layers.Conv2D(64,  (3, 3), activation="relu")(x)x = layers.MaxPooling2D()(x)x = layers.Conv2D(128, (3, 3), activation="relu")(x)x = layers.MaxPooling2D()(x)x = layers.Flatten()(x)x = layers.Dense(128, activation="relu")(x)output = layers.Dense(1, activation="sigmoid")(x)   # binary outputmy_model = tf.keras.Model(inputs=inputs, outputs=output)my_model.compile(    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),    loss="binary_crossentropy",    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])my_model.summary()

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────early_stop = EarlyStopping(    monitor='val_loss', patience=5,    min_delta=1e-7, restore_best_weights=True)reduce_lr = ReduceLROnPlateau(    monitor='val_loss', factor=0.2,    patience=2, min_delta=1e-7, verbose=1)checkpoint = ModelCheckpoint(    filepath='custom_cnn_v2.keras',    monitor='val_loss', save_best_only=True, verbose=1)

In [ ]:
# ── Class Weights (handle imbalance) ──────────────────────────────────────────class_weights_array = compute_class_weight(    class_weight='balanced',    classes=np.array([0, 1]),    y=train_data.classes)class_weights = {0: class_weights_array[0], 1: class_weights_array[1]}print("Class weights:", class_weights)

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────my_history = my_model.fit(    train_data,    epochs=30,    validation_data=val_data,    class_weight=class_weights,    callbacks=[early_stop, reduce_lr, checkpoint],    steps_per_epoch=len(train_df) // BATCH,    validation_steps=len(val_df)  // BATCH)

In [ ]:
# ── Evaluation helper (reused across all models) ─────────────────────────────def evaluation(history, y_act, y_prd):    """Plot loss/accuracy curves, confusion matrix, and classification metrics."""    tr_acc,  tr_loss  = history.history['accuracy'],     history.history['loss']    val_acc, val_loss = history.history['val_accuracy'], history.history['val_loss']    epochs = range(1, len(tr_acc) + 1)    # Training curves    plt.figure(figsize=(14, 5))    plt.subplot(1, 2, 1)    plt.plot(epochs, tr_loss,  'r', label='Train Loss')    plt.plot(epochs, val_loss, 'g', label='Val Loss')    plt.title('Loss'); plt.legend(); plt.xlabel('Epoch')    plt.subplot(1, 2, 2)    plt.plot(epochs, tr_acc,  'r', label='Train Acc')    plt.plot(epochs, val_acc, 'g', label='Val Acc')    plt.title('Accuracy'); plt.legend(); plt.xlabel('Epoch')    plt.tight_layout(); plt.show()    # Confusion matrix    plt.figure(figsize=(6, 5))    sns.heatmap(confusion_matrix(y_act, y_prd), annot=True, fmt="d",                cmap="Blues",                xticklabels=["NORMAL", "PNEUMONIA"],                yticklabels=["NORMAL", "PNEUMONIA"])    plt.title("Confusion Matrix")    plt.xlabel("Predicted"); plt.ylabel("Actual")    plt.show()    # Scalar metrics    print(f"Accuracy  : {accuracy_score(y_act,  y_prd):.4f}")    print(f"Precision : {precision_score(y_act, y_prd, pos_label=1):.4f}")    print(f"Recall    : {recall_score(y_act,    y_prd, pos_label=1):.4f}")    print(f"F1-Score  : {f1_score(y_act,        y_prd, pos_label=1):.4f}")

In [ ]:
# Predict on the test set (reset generator to ensure order)test_data.reset()y_true = np.array(test_data.classes)y_pred_probs = my_model.predict(test_data, steps=len(test_data))y_pred = (y_pred_probs > 0.5).astype(int).flatten()evaluation(my_history, y_true, y_pred)

## Phase 6 — DenseNet121 Transfer Learning (Frozen Base)We switch to **DenseNet121** pre-trained on ImageNet.  In Stage 1 the convolutional base is **fully frozen** — only the custom head learns.

In [ ]:
# ── DenseNet-specific generators (use DenseNet preprocessing, not simple /255) ─train_gen_dn = tf.keras.preprocessing.image.ImageDataGenerator(    preprocessing_function=preprocess_input,    zoom_range=0.1,    width_shift_range=0.1,    height_shift_range=0.1,    horizontal_flip=True)val_gen_dn = tf.keras.preprocessing.image.ImageDataGenerator(    preprocessing_function=preprocess_input)train_data_dn = train_gen_dn.flow_from_dataframe(    train_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=BATCH, color_mode='rgb')val_data_dn = val_gen_dn.flow_from_dataframe(    val_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=BATCH, color_mode='rgb')test_data_dn = val_gen_dn.flow_from_dataframe(    test_df, x_col='img_paths', y_col='label',    target_size=IMG_SIZE, class_mode='binary',    batch_size=1, color_mode='rgb', shuffle=False)

In [ ]:
# ── Model: DenseNet121 base + custom head ────────────────────────────────────base_model = tf.keras.applications.DenseNet121(    weights='imagenet',    include_top=False,    input_shape=(224, 224, 3))base_model.trainable = False   # freeze all base layers in Stage 1inputs  = tf.keras.Input(shape=(224, 224, 3))x       = base_model(inputs, training=False)   # training=False keeps BN in inference modex       = layers.GlobalAveragePooling2D()(x)x       = layers.Dense(128, activation="relu")(x)x       = layers.Dropout(0.3)(x)outputs = layers.Dense(1, activation="sigmoid")(x)dn_model = tf.keras.Model(inputs, outputs)dn_model.compile(    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5),    loss="binary_crossentropy",    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])dn_model.summary()

In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────────early_stop_dn = EarlyStopping(    monitor='val_loss', patience=5, restore_best_weights=True)reduce_lr_dn = ReduceLROnPlateau(    monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)checkpoint_dn = ModelCheckpoint(    filepath='densenet_stage1_v2.keras',    monitor='val_loss', save_best_only=True, verbose=1)

In [ ]:
# ── Stage 1 Training (frozen base) ───────────────────────────────────────────history_dn_stage1 = dn_model.fit(    train_data_dn,    epochs=20,    validation_data=val_data_dn,    class_weight=class_weights,    callbacks=[early_stop_dn, reduce_lr_dn, checkpoint_dn])

In [ ]:
# Stage 1 evaluationtest_data_dn.reset()y_true_dn       = np.array(test_data_dn.classes)y_pred_probs_dn = dn_model.predict(test_data_dn, steps=len(test_data_dn))y_pred_dn       = (y_pred_probs_dn > 0.5).astype(int).flatten()evaluation(history_dn_stage1, y_true_dn, y_pred_dn)

## Phase 7 — DenseNet121 Fine-Tuning (Unfreeze Top Layers)After the head has converged, we **unfreeze the last 13 layers** of the base  and train end-to-end with a very small learning rate to avoid destroying pretrained features.

In [ ]:
# Unfreeze only the top 13 layers of the base modelbase_model.trainable = Truefor layer in base_model.layers[:-13]:    layer.trainable = False# Recompile with a much lower learning rate for careful fine-tuningdn_model.compile(    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-6),    loss="binary_crossentropy",    metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])history_dn_ft = dn_model.fit(    train_data_dn,    epochs=30,    validation_data=val_data_dn,    class_weight=class_weights,    callbacks=[early_stop_dn, reduce_lr_dn, checkpoint_dn])

In [ ]:
# Fine-tuning evaluationtest_data_dn.reset()y_pred_probs_ft = dn_model.predict(test_data_dn, steps=len(test_data_dn))y_pred_ft       = (y_pred_probs_ft > 0.5).astype(int).flatten()evaluation(history_dn_ft, y_true_dn, y_pred_ft)

## Phase 8 — Final Evaluation & Model Export

In [ ]:
from sklearn import metrics# Full classification reportprint(metrics.classification_report(    y_true_dn, y_pred_ft,    target_names=["NORMAL", "PNEUMONIA"]))# ROC-AUCroc_auc = metrics.roc_auc_score(y_true_dn, y_pred_probs_ft)print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
# Save the final fine-tuned modeldn_model.save('pneumonia_densenet_stage1_final.keras')print("Model saved.")

In [ ]:
# List output files and provide a download link for the best checkpointprint(os.listdir('/kaggle/working'))from IPython.display import FileLinkFileLink('densenet_stage1_v2.keras')